# ZelusBench: Structural Attention — Diamond**A → B, A → C, D = f(B, C), then tail.**Fixed DAG structure, randomized background conditions(chain depth, distractors, transforms, dimensionality).Backgrounds are deterministic per scenario index — shared across all levels.- Structure: DIAMOND- 15 scenarios

In [ ]:
import kaggle_benchmarks as kbench
import numpy as np
import random
import re
import time

from zelusbench.scenarios.config import (
    ScenarioConfig, DAGStructure, QueryType,
    DistractorLevel, TransformDensity,
)
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.scorer import score_query, score_response




DAG = DAGStructure.DIAMOND
STRUCTURE_NAME = "DIAMOND"
SEEDS = 15

print(f"ZelusBench Structural Attention — Diamond")
print(f"Structure: {STRUCTURE_NAME} | Seeds: {SEEDS}")

In [ ]:
@kbench.task(name="zelusbench_attn_structural_diamond")
def zelusbench_attn_structural_diamond(llm) -> tuple[float, float]:
    _start = time.time()
    print(f"Running {SEEDS} scenarios (structure={STRUCTURE_NAME})...")
    print("=" * 60)

    all_scores = []

    for i in range(SEEDS):
        bg_rng = random.Random(i * 7919)
        cfg = ScenarioConfig.randomize_except(bg_rng, pinned={
            "dag_structure": DAG,
            "num_queries": 3,
            "seed": hash(STRUCTURE_NAME) % 10000 * 100 + i,
        })
        gen = ScenarioGenerator(cfg)
        scenario = gen.generate(f"structural_{STRUCTURE_NAME}_{i}")

        response = llm.prompt(scenario.prompt)
        scores = score_response(response, scenario)
        all_scores.extend(scores)

        avg = float(np.mean([s.score for s in scores]))
        q_depths = [s.chain_depth for s in scores]
        tiers = [s.tier.name for s in scores]
        bg = f"dim={cfg.dim} depth={cfg.min_chain_depth}-{cfg.max_chain_depth} dist={cfg.distractor_level.name} tx={cfg.transform_density.name}"
        print(f"  [{i+1}/{SEEDS}] avg={avg:.2f} q_depths={q_depths} tiers={tiers} | {bg}")

    overall = float(np.mean([s.score for s in all_scores]))
    std = float(np.std([s.score for s in all_scores]))
    elapsed = time.time() - _start
    print(f"\nOverall: {overall:.3f} +/- {std:.3f} | {len(all_scores)} queries | {elapsed:.1f}s")
    kbench.assertions.assert_true(overall >= 0, expectation=f"{STRUCTURE_NAME} valid (got {overall:.3f})")
    return overall, std


zelusbench_attn_structural_diamond.run(llm=kbench.llm)

In [ ]:
%choose zelusbench_attn_structural_diamond